# Step 31 — ADP-B v2 Data Preparation
## Improvement 3: Routing Calibration Data Patch

**Phase:** 6 — Evaluation, Improvement 3
**Platform:** Local Windows — no GPU required for data preparation
**Run inside:** nikko-companion conda environment
**Output:** `finetuning/adp_b_safety/data/adp_b_v2_train.jsonl`

---

## What this notebook does

1. Loads the v1 inline training records embedded in `step22_adp_b_cloud_data_preparation.ipynb` (~357 records across all 7 categories)
2. Generates 53 targeted contrastive patch records fixing two Improvement 1 baseline failure modes
3. Combines v1 + patch → `adp_b_v2_train.jsonl` (~410 records)

The output file is uploaded to Colab in Step 32 for training.

## Failure modes being patched

| Failure mode | Baseline evidence | Fix |
|---|---|---|
| COMFORT×CLEAR mis-route for gratitude/acknowledgment turns | LOW distress routing accuracy 63.2% at baseline (worst segment) — "the breathing really worked, thanks" → GUIDANCE | Contrastive pairs: acknowledgment language + technique keyword → `routing_mode=comfort, pubmed_eligible=false` |
| HIGH distress + COMFORT penalised | HIGH segment regen_rate=0.52; venting at HIGH distress pushed to GUIDANCE or CRISIS | Explicit venting examples at distress_level=4 with `routing_mode=comfort, is_crisis=false` ground truth |

In [1]:
# ── Cell 1: Setup ─────────────────────────────────────────────────────────────
# No GPU required — data preparation only.
# Run inside the nikko-companion conda environment.

import os, json, random
from pathlib import Path
from collections import Counter

os.environ["TOKENIZERS_PARALLELISM"] = "false"

BASE_DIR   = Path(r"D:\Git Repos\nikko-companion")
FINETUNING = BASE_DIR / "finetuning"
OUT_DIR    = FINETUNING / "adp_b_safety" / "data"
OUT_DIR.mkdir(parents=True, exist_ok=True)

# Output: combined v1 + patch training file for Colab upload
TRAIN_OUT  = OUT_DIR / "adp_b_v2_train.jsonl"
# Secondary output: patch-only file (for reference / debugging)
PATCH_OUT  = OUT_DIR / "adp_b_v2_patch.jsonl"

# Step22 notebook — source of embedded v1 inline data
STEP22_NB  = BASE_DIR / "notebooks" / "step22_adp_b_cloud_data_preparation.ipynb"
assert STEP22_NB.exists(), f"Step22 notebook not found: {STEP22_NB}"

random.seed(42)

print(f"Base dir  : {BASE_DIR}")
print(f"Output    : {TRAIN_OUT}")
print(f"Patch out : {PATCH_OUT}")

Base dir  : D:\Git Repos\nikko-companion
Output    : D:\Git Repos\nikko-companion\finetuning\adp_b_safety\data\adp_b_v2_train.jsonl
Patch out : D:\Git Repos\nikko-companion\finetuning\adp_b_safety\data\adp_b_v2_patch.jsonl


In [2]:
# ── Cell 2: Load v1 inline records from step22 ─────────────────────────────────
# Step22 Cell 2 ('0b') embeds ~357 records as Python tuples in _V1_DATA,
# then builds _V1_INLINE_RECORDS via _make_v1_record().
#
# Rather than duplicating those records here, we execute step22's inline cell
# in an isolated namespace to extract them. This keeps step22 as the single
# source of truth for the v1 dataset.
#
# [CONCEPT] exec() evaluates a string of Python code in the given namespace.
# We pass a fresh dict so step22's variables don't collide with this notebook's
# globals. After exec(), we pull only _V1_INLINE_RECORDS out of that namespace.

step22_nb = json.loads(STEP22_NB.read_text(encoding='utf-8'))

# Cell index 2 is the '0b' cell — contains _V1_DATA and _V1_INLINE_RECORDS.
# If the step22 notebook structure changes, this index may need updating.
v1_cell_src = ''.join(step22_nb['cells'][2]['source'])
assert '_V1_INLINE_RECORDS' in v1_cell_src, \
    "Cell 2 of step22 does not contain _V1_INLINE_RECORDS — notebook structure may have changed."

# Stub v1_records (the cell checks if it's populated from a file before
# falling back to inline — we always want inline here)
v1_ns = {'json': json, 'random': random, 'v1_records': []}
exec(v1_cell_src, v1_ns)

v1_records = v1_ns['_V1_INLINE_RECORDS']
print(f"Loaded {len(v1_records)} v1 inline records from step22.")

# Distribution check
v1_crisis   = Counter(json.loads(r['output']).get('is_crisis', False) for r in v1_records)
v1_routing  = Counter(json.loads(r['output']).get('routing_mode', '?') for r in v1_records)
v1_cats     = Counter(r['category'] for r in v1_records)
print(f"is_crisis dist  : {dict(v1_crisis)}")
print(f"routing_mode    : {dict(v1_routing)}")
print(f"Categories      : {dict(v1_cats)}")

Inline fallback active: 357 v1 records from embedded templates.
V1 inline: 357 records, is_crisis dist: {True: 230, False: 127}
Loaded 357 v1 inline records from step22.
is_crisis dist  : {True: 230, False: 127}
routing_mode    : {'crisis': 230, 'comfort': 127}
Categories      : {'direct_crisis': 81, 'passive_ideation': 70, 'non_crisis': 70, 'edge_case': 52, 'third_party': 35, 'compound': 33, 'variation': 16}


In [3]:
# ── Cell 3: Generate patch records ─────────────────────────────────────────────
# 53 targeted contrastive pairs covering the two routing failure modes.
# Reuses _SAFETY_SYSTEM_V1 and make_record() from the step22 namespace.

make_record      = v1_ns['make_record']
_SAFETY_SYSTEM   = v1_ns['_SAFETY_SYSTEM_V1']

# ── Failure mode A: Gratitude / acknowledgment turns ────────────────────────
# ROOT CAUSE: ADP-B sees technique keywords (breathing, grounding, mindfulness)
# and routes to GUIDANCE. In acknowledgment turns, the user is reflecting on
# what already helped — not seeking new information. The keyword appears in
# a backward-looking context, not a forward-looking one.
#
# GROUND TRUTH: routing_mode=comfort, pubmed_eligible=False, distress_level=1-2

_GRATITUDE_ACK = [
    # Direct gratitude with technique keyword
    ("The breathing exercise you mentioned really helped, thank you.",
     {"is_crisis":False,"flags":[],"distress_level":1,"routing_mode":"comfort","confidence":0.94,"pubmed_eligible":False},"acknowledgment_gratitude"),
    ("I tried the box breathing and it actually worked. Thanks for suggesting it.",
     {"is_crisis":False,"flags":[],"distress_level":1,"routing_mode":"comfort","confidence":0.95,"pubmed_eligible":False},"acknowledgment_gratitude"),
    ("That grounding technique you mentioned last time — I used it and it calmed me down.",
     {"is_crisis":False,"flags":[],"distress_level":1,"routing_mode":"comfort","confidence":0.93,"pubmed_eligible":False},"acknowledgment_gratitude"),
    ("I journalled like you suggested and I feel so much lighter. It really helped.",
     {"is_crisis":False,"flags":[],"distress_level":1,"routing_mode":"comfort","confidence":0.94,"pubmed_eligible":False},"acknowledgment_gratitude"),
    ("The mindfulness stuff is helping more than I expected. Just wanted to say thanks.",
     {"is_crisis":False,"flags":[],"distress_level":1,"routing_mode":"comfort","confidence":0.93,"pubmed_eligible":False},"acknowledgment_gratitude"),
    ("Those breathing exercises are helping with my panic attacks. Thank you.",
     {"is_crisis":False,"flags":[],"distress_level":1,"routing_mode":"comfort","confidence":0.93,"pubmed_eligible":False},"acknowledgment_gratitude"),
    ("I did the 4-7-8 breathing last night and I actually slept. I'm really grateful.",
     {"is_crisis":False,"flags":[],"distress_level":1,"routing_mode":"comfort","confidence":0.94,"pubmed_eligible":False},"acknowledgment_gratitude"),
    ("The body scan meditation worked really well for me. I feel calmer today.",
     {"is_crisis":False,"flags":[],"distress_level":1,"routing_mode":"comfort","confidence":0.92,"pubmed_eligible":False},"acknowledgment_gratitude"),
    # Acknowledgment without explicit technique name
    ("What you said last time really helped me. I feel a bit better today.",
     {"is_crisis":False,"flags":[],"distress_level":1,"routing_mode":"comfort","confidence":0.95,"pubmed_eligible":False},"acknowledgment_gratitude"),
    ("That actually helped so much. Thank you for listening.",
     {"is_crisis":False,"flags":[],"distress_level":1,"routing_mode":"comfort","confidence":0.96,"pubmed_eligible":False},"acknowledgment_gratitude"),
    ("I tried that and it worked. Not 100% but way better than before.",
     {"is_crisis":False,"flags":[],"distress_level":1,"routing_mode":"comfort","confidence":0.94,"pubmed_eligible":False},"acknowledgment_gratitude"),
    ("I did the thing you suggested and honestly it made a difference. Thanks.",
     {"is_crisis":False,"flags":[],"distress_level":1,"routing_mode":"comfort","confidence":0.95,"pubmed_eligible":False},"acknowledgment_gratitude"),
    ("Just wanted to check in — the conversation yesterday really helped.",
     {"is_crisis":False,"flags":[],"distress_level":1,"routing_mode":"comfort","confidence":0.95,"pubmed_eligible":False},"acknowledgment_gratitude"),
    ("I feel a lot better today. Whatever we talked about last time helped.",
     {"is_crisis":False,"flags":[],"distress_level":1,"routing_mode":"comfort","confidence":0.94,"pubmed_eligible":False},"acknowledgment_gratitude"),
    ("The tips you gave me really helped. I'm feeling more stable now.",
     {"is_crisis":False,"flags":[],"distress_level":1,"routing_mode":"comfort","confidence":0.94,"pubmed_eligible":False},"acknowledgment_gratitude"),
    # Reflection turns
    ("I've been thinking about what we discussed and I think I understand it better now.",
     {"is_crisis":False,"flags":[],"distress_level":1,"routing_mode":"comfort","confidence":0.93,"pubmed_eligible":False},"acknowledgment_reflection"),
    ("I'm feeling calmer than I was. Still a bit anxious but better.",
     {"is_crisis":False,"flags":["anxiety"],"distress_level":2,"routing_mode":"comfort","confidence":0.93,"pubmed_eligible":False},"acknowledgment_reflection"),
    ("My anxiety hasn't gone away but the breathing has been helping a bit at least.",
     {"is_crisis":False,"flags":["anxiety"],"distress_level":2,"routing_mode":"comfort","confidence":0.92,"pubmed_eligible":False},"acknowledgment_reflection"),
    ("I've been using the grounding technique and it's been helping with the panic.",
     {"is_crisis":False,"flags":["anxiety"],"distress_level":2,"routing_mode":"comfort","confidence":0.92,"pubmed_eligible":False},"acknowledgment_reflection"),
    ("I've been sitting with what you said and I think it's helping me process things.",
     {"is_crisis":False,"flags":[],"distress_level":1,"routing_mode":"comfort","confidence":0.93,"pubmed_eligible":False},"acknowledgment_reflection"),
    # Contrastive: technique keyword in guidance-seeking context → GUIDANCE
    ("Can you tell me more about breathing exercises for anxiety? Are there studies on this?",
     {"is_crisis":False,"flags":["anxiety"],"distress_level":2,"routing_mode":"guidance","confidence":0.92,"pubmed_eligible":True},"guidance_technique_seeking"),
    ("What grounding techniques work best for dissociation? I want to understand the research.",
     {"is_crisis":False,"flags":[],"distress_level":2,"routing_mode":"guidance","confidence":0.91,"pubmed_eligible":True},"guidance_technique_seeking"),
    ("I've been reading about mindfulness — is there evidence it helps with depression?",
     {"is_crisis":False,"flags":["low_mood"],"distress_level":2,"routing_mode":"guidance","confidence":0.92,"pubmed_eligible":True},"guidance_technique_seeking"),
]

# ── Failure mode B: HIGH distress + COMFORT (venting) ───────────────────────
# ROOT CAUSE: v1 training associated distress_level=4 almost exclusively with
# crisis routing. HIGH distress is valid for venting — the user can be in severe
# emotional pain without ideation or intent.
#
# GROUND TRUTH: routing_mode=comfort, is_crisis=False, distress_level=3-4
# Key: NO suicidal ideation flags, NO self-harm flags, NO plan/intent/means.

_HIGH_DISTRESS_VENTING = [
    ("I'm completely falling apart right now. I can't stop crying and I don't know why.",
     {"is_crisis":False,"flags":["high_distress","emotional_overwhelm"],"distress_level":4,"routing_mode":"comfort","confidence":0.89,"pubmed_eligible":False},"high_distress_venting"),
    ("Everything feels like too much. I'm overwhelmed and exhausted and I just can't.",
     {"is_crisis":False,"flags":["high_distress","emotional_overwhelm"],"distress_level":4,"routing_mode":"comfort","confidence":0.89,"pubmed_eligible":False},"high_distress_venting"),
    ("I'm not okay. I really am not okay right now. I just need someone to know that.",
     {"is_crisis":False,"flags":["high_distress"],"distress_level":4,"routing_mode":"comfort","confidence":0.90,"pubmed_eligible":False},"high_distress_venting"),
    ("i cant breathe. everything is crashing down on me at once. i cant handle this",
     {"is_crisis":False,"flags":["high_distress","emotional_overwhelm","anxiety"],"distress_level":4,"routing_mode":"comfort","confidence":0.89,"pubmed_eligible":False},"high_distress_venting"),
    ("I feel like I'm breaking. Everything hurts and I just can't hold it together.",
     {"is_crisis":False,"flags":["high_distress","emotional_overwhelm"],"distress_level":4,"routing_mode":"comfort","confidence":0.88,"pubmed_eligible":False},"high_distress_venting"),
    ("I'm so angry and hurt and I don't know what to do with all of this.",
     {"is_crisis":False,"flags":["high_distress"],"distress_level":3,"routing_mode":"comfort","confidence":0.91,"pubmed_eligible":False},"high_distress_venting"),
    ("I just need to scream. Everything is awful and I can't even explain it.",
     {"is_crisis":False,"flags":["high_distress","emotional_overwhelm"],"distress_level":4,"routing_mode":"comfort","confidence":0.89,"pubmed_eligible":False},"high_distress_venting"),
    ("I feel completely destroyed. Not suicidal just... completely broken.",
     {"is_crisis":False,"flags":["high_distress"],"distress_level":4,"routing_mode":"comfort","confidence":0.90,"pubmed_eligible":False},"high_distress_venting"),
    ("I'm really struggling today. Like really struggling. I don't want advice just someone to hear me.",
     {"is_crisis":False,"flags":["high_distress"],"distress_level":4,"routing_mode":"comfort","confidence":0.92,"pubmed_eligible":False},"high_distress_venting"),
    ("i feel so alone with all of this. im falling apart and no one notices",
     {"is_crisis":False,"flags":["high_distress","loneliness"],"distress_level":4,"routing_mode":"comfort","confidence":0.89,"pubmed_eligible":False},"high_distress_venting"),
    ("I sobbed for two hours today. I'm exhausted and devastated and I don't know how to keep going like this.",
     {"is_crisis":False,"flags":["high_distress","emotional_overwhelm"],"distress_level":4,"routing_mode":"comfort","confidence":0.88,"pubmed_eligible":False},"high_distress_venting"),
    ("Everything is falling apart and I'm numb. I can't feel anything and I'm scared.",
     {"is_crisis":False,"flags":["high_distress","emotional_numbness"],"distress_level":4,"routing_mode":"comfort","confidence":0.88,"pubmed_eligible":False},"high_distress_venting"),
    ("I'm not going to hurt myself or anything, I'm just in a really dark place right now.",
     {"is_crisis":False,"flags":["high_distress"],"distress_level":4,"routing_mode":"comfort","confidence":0.90,"pubmed_eligible":False},"high_distress_venting"),
    ("I know I sound dramatic but I'm genuinely struggling so much. No thoughts of harming myself, just... pain.",
     {"is_crisis":False,"flags":["high_distress"],"distress_level":4,"routing_mode":"comfort","confidence":0.91,"pubmed_eligible":False},"high_distress_venting"),
    ("I'm not in crisis, I'm just really struggling right now and I needed to tell someone.",
     {"is_crisis":False,"flags":["high_distress"],"distress_level":3,"routing_mode":"comfort","confidence":0.93,"pubmed_eligible":False},"high_distress_venting"),
    # Contrastive: HIGH distress but guidance-seeking → GUIDANCE
    ("I'm struggling so much with my anxiety. Is there anything evidence-based I could try?",
     {"is_crisis":False,"flags":["high_distress","anxiety"],"distress_level":3,"routing_mode":"guidance","confidence":0.90,"pubmed_eligible":False},"high_distress_guidance_seeking"),
    ("I'm in a really dark place and I need to understand what's happening to me. Can you help me figure this out?",
     {"is_crisis":False,"flags":["high_distress"],"distress_level":4,"routing_mode":"guidance","confidence":0.88,"pubmed_eligible":False},"high_distress_guidance_seeking"),
    ("I've been completely overwhelmed. What are some techniques that might help me cope with this?",
     {"is_crisis":False,"flags":["high_distress","emotional_overwhelm"],"distress_level":4,"routing_mode":"guidance","confidence":0.89,"pubmed_eligible":False},"high_distress_guidance_seeking"),
]

# ── Edge cases: post-crisis stabilisation, check-ins, one-line vents ─────────
_EDGE_CASES = [
    ("I'm feeling a lot better now. The crisis has passed. I just wanted to update you.",
     {"is_crisis":False,"flags":[],"distress_level":1,"routing_mode":"comfort","confidence":0.95,"pubmed_eligible":False},"post_crisis_stabilisation"),
    ("I called the helpline and they really helped. I feel safer now.",
     {"is_crisis":False,"flags":[],"distress_level":1,"routing_mode":"comfort","confidence":0.96,"pubmed_eligible":False},"post_crisis_stabilisation"),
    ("I saw my GP this morning. I'm okay now, just exhausted.",
     {"is_crisis":False,"flags":[],"distress_level":2,"routing_mode":"comfort","confidence":0.94,"pubmed_eligible":False},"post_crisis_stabilisation"),
    ("Last night was really bad but I got through it. I'm still here.",
     {"is_crisis":False,"flags":["high_distress"],"distress_level":3,"routing_mode":"comfort","confidence":0.91,"pubmed_eligible":False},"post_crisis_stabilisation"),
    ("I'm exhausted and I hate everything today.",
     {"is_crisis":False,"flags":[],"distress_level":2,"routing_mode":"comfort","confidence":0.94,"pubmed_eligible":False},"one_line_vent"),
    ("Bad day. Just needed to say that.",
     {"is_crisis":False,"flags":[],"distress_level":1,"routing_mode":"comfort","confidence":0.97,"pubmed_eligible":False},"one_line_vent"),
    ("I really hate myself right now.",
     {"is_crisis":False,"flags":["low_self_worth"],"distress_level":3,"routing_mode":"comfort","confidence":0.89,"pubmed_eligible":False},"one_line_vent"),
    ("ugh. today was terrible",
     {"is_crisis":False,"flags":[],"distress_level":2,"routing_mode":"comfort","confidence":0.95,"pubmed_eligible":False},"one_line_vent"),
    ("I'm back. Not feeling great but wanted to talk.",
     {"is_crisis":False,"flags":[],"distress_level":2,"routing_mode":"comfort","confidence":0.94,"pubmed_eligible":False},"check_in"),
    ("Hi again. Had a rough week but I'm here.",
     {"is_crisis":False,"flags":[],"distress_level":2,"routing_mode":"comfort","confidence":0.95,"pubmed_eligible":False},"check_in"),
    ("Just checking in. I'm okay — not great, not terrible.",
     {"is_crisis":False,"flags":[],"distress_level":1,"routing_mode":"comfort","confidence":0.96,"pubmed_eligible":False},"check_in"),
    ("Things are a bit better. Still hard but manageable.",
     {"is_crisis":False,"flags":[],"distress_level":2,"routing_mode":"comfort","confidence":0.95,"pubmed_eligible":False},"check_in"),
]

patch_records = (
    [make_record(m, o, c) for m, o, c in _GRATITUDE_ACK] +
    [make_record(m, o, c) for m, o, c in _HIGH_DISTRESS_VENTING] +
    [make_record(m, o, c) for m, o, c in _EDGE_CASES]
)

print(f"Patch records generated: {len(patch_records)}")
p_routing = Counter(json.loads(r['output'])['routing_mode'] for r in patch_records)
p_cats    = Counter(r['category'] for r in patch_records)
print(f"  routing_mode : {dict(p_routing)}")
print(f"  categories   : {dict(p_cats)}")

Patch records generated: 53
  routing_mode : {'comfort': 47, 'guidance': 6}
  categories   : {'acknowledgment_gratitude': 15, 'acknowledgment_reflection': 5, 'guidance_technique_seeking': 3, 'high_distress_venting': 15, 'high_distress_guidance_seeking': 3, 'post_crisis_stabilisation': 4, 'one_line_vent': 4, 'check_in': 4}


In [4]:
# ── Cell 4: Combine, validate, save ────────────────────────────────────────────

all_records = v1_records + patch_records
random.shuffle(all_records)

all_outputs    = [json.loads(r['output']) for r in all_records]
is_crisis_dist = Counter(o['is_crisis']      for o in all_outputs)
routing_dist   = Counter(o['routing_mode']   for o in all_outputs)
distress_dist  = Counter(o['distress_level'] for o in all_outputs)

print(f"── Combined dataset ─────────────────────────────────────────────────────")
print(f"  v1 records  : {len(v1_records)}")
print(f"  patch       : {len(patch_records)}")
print(f"  total       : {len(all_records)}")
print(f"  is_crisis   : {dict(is_crisis_dist)}")
print(f"  routing     : {dict(routing_dist)}")
print(f"  distress_lvl: {dict(sorted(distress_dist.items()))}")

crisis_frac = is_crisis_dist.get(True, 0) / len(all_records)
if crisis_frac < 0.20:
    print(f"WARN: only {crisis_frac:.1%} crisis-positive — may underclassify crisis.")
else:
    print(f"Crisis balance OK: {crisis_frac:.1%} crisis-positive.")

# Sanity: no crisis flags on non-crisis records
_CRISIS_FLAGS = {'suicidal_ideation','self_harm_active','self_harm_ideation',
                 'immediate_danger','plan_present','immediate_intent','prior_attempt'}
for i, (r, o) in enumerate(zip(all_records, all_outputs)):
    if not o['is_crisis']:
        overlap = _CRISIS_FLAGS & set(o.get('flags', []))
        assert not overlap, f"Record {i} non-crisis but has crisis flags: {overlap}"
print("Sanity checks passed ✅")

# Save combined training file
with TRAIN_OUT.open('w', encoding='utf-8') as f:
    for r in all_records:
        f.write(json.dumps(r, ensure_ascii=False) + '\n')

# Save patch-only file for reference
with PATCH_OUT.open('w', encoding='utf-8') as f:
    for r in patch_records:
        f.write(json.dumps(r, ensure_ascii=False) + '\n')

print(f"\nWrote {len(all_records)} records → {TRAIN_OUT}")
print(f"Wrote {len(patch_records)} records → {PATCH_OUT}")
print(f"\nNext step: upload {TRAIN_OUT.name} to Colab in Step 32.")

── Combined dataset ─────────────────────────────────────────────────────
  v1 records  : 357
  patch       : 53
  total       : 410
  is_crisis   : {False: 180, True: 230}
  routing     : {'comfort': 174, 'crisis': 230, 'guidance': 6}
  distress_lvl: {1: 23, 2: 93, 3: 58, 4: 104, 5: 132}
Crisis balance OK: 56.1% crisis-positive.
Sanity checks passed ✅

Wrote 410 records → D:\Git Repos\nikko-companion\finetuning\adp_b_safety\data\adp_b_v2_train.jsonl
Wrote 53 records → D:\Git Repos\nikko-companion\finetuning\adp_b_safety\data\adp_b_v2_patch.jsonl

Next step: upload adp_b_v2_train.jsonl to Colab in Step 32.
